# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

,TL,TM,TR,ML,MM,MR,BL,BM,BR,class
0,x,x,x,x,o,o,x,o,o,True
1,x,x,x,x,o,o,o,x,o,True
2,x,x,x,x,o,o,o,o,x,True
3,x,x,x,x,o,o,o,b,b,True
4,x,x,x,x,o,o,b,o,b,True


In [5]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Read
url = "https://raw.githubusercontent.com/roberto-vargas-c/lab-neural-networks/master/your-code/tic-tac-toe.csv"
df = pd.read_csv(url)
df.head()

# 2. Inspect
print(df.shape); print(df.head()); print(df.info())
print(df.isna().sum().sum(), df.duplicated().sum())
for c in df.columns: print(c, df[c].unique())
print(df["class"].value_counts())

# 3. Categorical -> numeric
feature_cols = [c for c in df.columns if c != "class"]
board_map = {"x": 1, "o": -1, "b": 0}
df[feature_cols] = df[feature_cols].replace(board_map).astype(int)
df["class"] = df["class"].astype(int)

# 4. Inputs / output
X = df[feature_cols]
y = df["class"]

# 5. Normalize
X_norm = pd.DataFrame(MinMaxScaler().fit_transform(X), columns=feature_cols)

(958, 10)
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   TL      958 non-null    object
 1   TM      958 non-null    object
 2   TR      958 non-null    object
 3   ML      958 non-null    object
 4   MM      958 non-null    object
 5   MR      958 non-null    object
 6   BL      958 non-null    object
 7   BM      958 non-null    object
 8   BR      958 non-null    object
 9   class   958 non-null    bool  
dtypes: bool(1), object(9)
memory usage: 68.4+ KB
None
0 0
TL ['x' 'o' 'b']
TM ['x' 'o' 'b']
TR ['x' 'o' 'b']
ML ['x' 'o' 'b']
MM ['o' 'b' 'x']
MR ['o' 'b' 'x']
BL ['x' 'o' 'b']
BM ['o' 'x' 'b']
BR ['o' 'x' 'b']
class [ True False]
class

/tmp/ipykernel_10061/2866509061.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[feature_cols] = df[feature_cols].replace(board_map).astype(int)


## Step 2: Build Neural Network

To build the neural network, you can refer to your own codes you wrote while following the [Deep Learning with Python, TensorFlow, and Keras tutorial](https://www.youtube.com/watch?v=wQ8BIBpya2k) in the lesson. It's pretty similar to what you will be doing in this lab.

1. Split the training and test data.
1. Create a `Sequential` model.
1. Add several layers to your model. Make sure you use ReLU as the activation function for the middle layers. Use Softmax for the output layer because each output has a single lable and all the label probabilities add up to 1.
1. Compile the model using `adam` as the optimizer and `sparse_categorical_crossentropy` as the loss function. For metrics, use `accuracy` for now.
1. Fit the training data.
1. Evaluate your neural network model with the test data.
1. Save your model as `tic-tac-toe.model`.

In [6]:
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

# 1. Split training and test data
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

# 2-3. Sequential model: ReLU in the middle, Softmax output
model = keras.Sequential([
    layers.Input(shape=(9,)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(2, activation="softmax"),   # 2 units: class 0 (O/no win) and 1 (X wins)
])

# 4. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# 5. Fit
history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

# 6. Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test loss: {test_loss:.4f}, test accuracy: {test_acc:.4f}")

# 7. Save
model.save("tic-tac-toe.keras")

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.6438 - loss: 0.6648 - val_accuracy: 0.6364 - val_loss: 0.6475
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6585 - loss: 0.6271 - val_accuracy: 0.6364 - val_loss: 0.6323
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6585 - loss: 0.6088 - val_accuracy: 0.6364 - val_loss: 0.6184
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6585 - loss: 0.5920 - val_accuracy: 0.6429 - val_loss: 0.6011
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6585 - loss: 0.5756 - val_accuracy: 0.6688 - val_loss: 0.5863
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7026 - loss: 0.5570 - val_accuracy: 0.7078 - val_loss: 0.5723
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7631 - loss: 0.5427 - val_accuracy: 0.7403 - val_loss: 0.5609
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7729 - loss: 0.5293 - val_accuracy: 0.7532 - 

## Step 3: Make Predictions

Now load your saved model and use it to make predictions on a few random rows in the test dataset. Check if the predictions are correct.

In [ ]:
import numpy as np

# Load the saved model (use whatever filename/path you saved with in Step 2)
loaded_model = keras.models.load_model("tic-tac-toe.keras")

# Pick a few random rows from the test set
sample = X_test.sample(5, random_state=1)
true_labels = y_test.loc[sample.index]

# Predict
probs = loaded_model.predict(sample)
pred_labels = np.argmax(probs, axis=1)

# Compare
results = sample.copy()
results["P(no win)"] = probs[:, 0].round(3)
results["P(X wins)"] = probs[:, 1].round(3)
results["predicted"] = pred_labels
results["actual"] = true_labels.values
results["correct"] = results["predicted"] == results["actual"]
print(results[["P(no win)", "P(X wins)", "predicted", "actual", "correct"]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 323ms/step
     P(no win)  P(X wins)  predicted  actual  correct
684      1.000      0.000          0       0     True
929      0.998      0.002          0       0     True
46       0.000      1.000          1       1     True
22       0.064      0.936          1       1     True
895      1.000      0.000          0       0     True


## Step 4: Improve Your Model

Did your model achieve low loss (<0.1) and high accuracy (>0.95)? If not, try to improve your model.

But how? There are so many things you can play with in Tensorflow and in the next challenge you'll learn about these things. But in this challenge, let's just do a few things to see if they will help.

* Add more layers to your model. If the data are complex you need more layers. But don't use more layers than you need. If adding more layers does not improve the model performance you don't need additional layers.
* Adjust the learning rate when you compile the model. This means you will create a custom `tf.keras.optimizers.Adam` instance where you specify the learning rate you want. Then pass the instance to `model.compile` as the optimizer.
    * `tf.keras.optimizers.Adam` [reference](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam).
    * Don't worry if you don't understand what the learning rate does. You'll learn about it in the next challenge.
* Adjust the number of epochs when you fit the training data to the model. Your model performance continues to improve as you train more epochs. But eventually it will reach the ceiling and the performance will stay the same.

In [9]:
# Reload raw data (so you start clean)
df = pd.read_csv(url)   # same URL as before

# One-hot encode the squares (27 columns), no scaling needed
X = pd.get_dummies(df.drop(columns="class"), dtype=int)
y = df["class"].astype(int)
print(X.shape)          # should be (958, 27)

# Same split as before, so results are comparable
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Rebuild the model with a 27-feature input
model = keras.Sequential([
    layers.Input(shape=(27,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(2, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

history = model.fit(X_train, y_train,
                    epochs=500, batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop], verbose=0)

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}, test accuracy: {test_acc:.4f}")
print("Stopped at epoch:", len(history.history["loss"]))

# Save
model.save("tic-tac-toe.keras")

(958, 27)
Test loss: 0.0242, test accuracy: 0.9896
Stopped at epoch: 183


**Which approach(es) did you find helpful to improve your model performance?**

In [11]:
# Encoding: one-hot (27 inputs) instead of the single numeric column per square.
# Architecture: 64-64-32 hidden units instead of 32-16.
# Training: the early stopping callback above.